# Demo 1: From Shape to Measurements

This notebook recreates the Task 2 MATLAB walkthrough in Python, using the same structure, comments, and computational steps shown in the source document.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


## Helper Function

In [ ]:
# Helper function translated directly from the MATLAB example
def compute_shape_mask(shape, X, Y):
    if shape["type"] == "ellipse":
        x_rot = (X - shape["center"][0]) * np.cos(shape["theta"]) + (Y - shape["center"][1]) * np.sin(shape["theta"])
        y_rot = -(X - shape["center"][0]) * np.sin(shape["theta"]) + (Y - shape["center"][1]) * np.cos(shape["theta"])
        mask = (x_rot / shape["a"]) ** 2 + (y_rot / shape["b"]) ** 2 <= 1.0
        return mask
    raise ValueError(f"Unsupported shape type: {shape['type']}")

## DEMO 1: Computing Gradient Measurements from a Shape

In [ ]:
# Configuration
N = 4  # Number of Fourier coefficients minus 1
num_measure_points = N + 1  # Number of measurement points

# Create a test shape
D = {
    "type": "ellipse",
    "a": 0.6,
    "b": 0.3,
    "theta": np.pi / 4,
    "center": (0.1, 0.1),
}

# Setup grid for coefficient computation
grid_size = 200
x = np.linspace(-1.0, 1.0, grid_size)
X, Y = np.meshgrid(x, x)
dx = x[1] - x[0]
dA = dx ** 2

# Compute mask and Fourier coefficients
mask = compute_shape_mask(D, X, Y)
z = X + 1j * Y

coefficients = np.zeros(N + 1, dtype=np.complex128)
for n in range(N + 1):
    integrand = (z ** n) * mask
    coefficients[n] = (1 / (4 * np.pi)) * np.sum(integrand) * dA

# Define measurement points on unit circle
theta = np.linspace(0.0, 2 * np.pi, num_measure_points + 1)
theta = theta[:-1]
measure_points = np.exp(1j * theta)

# Compute gradient at each measurement point
grad_data = np.zeros(num_measure_points, dtype=np.complex128)
for i, x_pt in enumerate(measure_points):
    grad = 0.0j
    for n in range(N + 1):
        grad = grad + 2 * np.conj(coefficients[n]) * x_pt ** (n + 1)
    grad_data[i] = grad

# Convert to real representation (for neural network input)
grad_data_real = np.concatenate([np.real(grad_data), np.imag(grad_data)])

## Visualization

In [ ]:
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 3, 1)
ax1.imshow(mask.astype(float), extent=[x.min(), x.max(), x.min(), x.max()], origin="lower", cmap="gray")
ax1.set_aspect("equal")
ax1.set_title("Original Shape")
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2 = fig.add_subplot(1, 3, 2)
# Plot measurement points on unit circle
ax2.plot(np.real(measure_points), np.imag(measure_points), "ro", markersize=12, linewidth=2)
ax2.plot(np.cos(np.linspace(0, 2 * np.pi, 100)), np.sin(np.linspace(0, 2 * np.pi, 100)), "k--", linewidth=1.5)
ax2.set_aspect("equal")
ax2.grid(True)
ax2.set_xlabel("Real")
ax2.set_ylabel("Imag")
ax2.set_title("Measurement Points on Unit Circle")
ax2.legend(["Measurement Points", "Unit Circle"], loc="best")

ax3 = fig.add_subplot(1, 3, 3)
# Show gradient magnitude at each measurement point
ax3.stem(np.arange(1, num_measure_points + 1), np.abs(grad_data), linefmt="b-", markerfmt="bo", basefmt=" ")
ax3.set_xlabel("Measurement Point Index")
ax3.set_ylabel("|Gradient|")
ax3.set_title("Gradient Magnitude at Measurement Points")
ax3.grid(True)

fig.suptitle("From Shape to Gradient Measurements", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Measurement Results

In [ ]:
measurement_table = pd.DataFrame({
    "Measurement Point": np.arange(1, num_measure_points + 1),
    "Angle (deg)": np.degrees(theta),
    "Gradient Magnitude": np.abs(grad_data),
})

print("\n=== MEASUREMENT RESULTS ===\n")
print(measurement_table.to_string(index=False, formatters={
    "Angle (deg)": lambda value: f"{value:.1f}",
    "Gradient Magnitude": lambda value: f"{value:.4f}",
}))

measurement_table

## Coefficient Table

In [ ]:
coefficient_table = pd.DataFrame({
    "n": np.arange(N + 1),
    "Real": np.real(coefficients),
    "Imaginary": np.imag(coefficients),
    "Magnitude": np.abs(coefficients),
})

print("\n=== FOURIER COEFFICIENTS ===\n")
print(coefficient_table.to_string(index=False, formatters={
    "Real": lambda value: f"{value:+.4f}",
    "Imaginary": lambda value: f"{value:+.4f}",
    "Magnitude": lambda value: f"{value:.4f}",
}))

coefficient_table

# Stage 1 - The DNN Coefficient Predictor

## DNN Architecture

In [ ]:
# Python equivalent of the MATLAB layer array
input_size = grad_data_real.shape[0]
output_size = 3

architecture_model = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, output_size),
)

print(architecture_model)
print('Parameter count:', sum(param.numel() for param in architecture_model.parameters()))


## TUTORIAL: Training Your First DNN

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# ===== STEP 1: Create simple synthetic data =====
print('STEP 1: Creating training data...\n')

num_samples = 1000
N_train = 3

a_vals = 0.3 + 0.5 * np.random.rand(num_samples, 1)
b_vals = 0.3 + 0.5 * np.random.rand(num_samples, 1)
theta_vals = np.pi * np.random.rand(num_samples, 1)

coeffs_real = np.zeros((num_samples, 2 * (N_train + 1)), dtype=np.float32)
for i in range(num_samples):
    coeffs_complex = np.zeros(N_train + 1, dtype=np.complex128)
    coeffs_complex[0] = a_vals[i, 0] * b_vals[i, 0]
    for n in range(1, N_train + 1):
        coeffs_complex[n] = (a_vals[i, 0] - b_vals[i, 0]) * np.exp(1j * theta_vals[i, 0]) * (0.5 ** n)
    coeffs_real[i, :] = np.concatenate([np.real(coeffs_complex), np.imag(coeffs_complex)]).astype(np.float32)

train_idx = np.arange(0, 800)
val_idx = np.arange(800, 1000)
X_train = coeffs_real[train_idx, :]
Y_train = np.hstack([a_vals[train_idx], b_vals[train_idx], theta_vals[train_idx]]).astype(np.float32)
X_val = coeffs_real[val_idx, :]
Y_val = np.hstack([a_vals[val_idx], b_vals[val_idx], theta_vals[val_idx]]).astype(np.float32)

X_mu = X_train.mean(axis=0, keepdims=True)
X_sigma = X_train.std(axis=0, keepdims=True)
X_sigma = np.where(X_sigma == 0, 1.0, X_sigma)
X_train_norm = (X_train - X_mu) / X_sigma
X_val_norm = (X_val - X_mu) / X_sigma

Y_mu = Y_train.mean(axis=0, keepdims=True)
Y_sigma = Y_train.std(axis=0, keepdims=True)
Y_sigma = np.where(Y_sigma == 0, 1.0, Y_sigma)
Y_train_norm = (Y_train - Y_mu) / Y_sigma
Y_val_norm = (Y_val - Y_mu) / Y_sigma

training_model = nn.Sequential(
    nn.Linear(X_train.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, Y_train.shape[1]),
)
optimizer = torch.optim.Adam(training_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
loader = DataLoader(
    TensorDataset(torch.tensor(X_train_norm, dtype=torch.float32), torch.tensor(Y_train_norm, dtype=torch.float32)),
    batch_size=32,
    shuffle=True,
)

print('STEP 5: Training network...\n')
history = {'loss': [], 'val_loss': [], 'mae': [], 'val_mae': []}
X_val_t = torch.tensor(X_val_norm, dtype=torch.float32)
Y_val_t = torch.tensor(Y_val_norm, dtype=torch.float32)
for epoch in range(50):
    training_model.train()
    epoch_losses = []
    epoch_maes = []
    for batch_X, batch_Y in loader:
        optimizer.zero_grad()
        preds = training_model(batch_X)
        loss = criterion(preds, batch_Y)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.item()))
        epoch_maes.append(float(torch.mean(torch.abs(preds - batch_Y)).item()))
    training_model.eval()
    with torch.no_grad():
        val_preds = training_model(X_val_t)
        val_loss = float(criterion(val_preds, Y_val_t).item())
        val_mae = float(torch.mean(torch.abs(val_preds - Y_val_t)).item())
    history['loss'].append(float(np.mean(epoch_losses)))
    history['mae'].append(float(np.mean(epoch_maes)))
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    print(f'Epoch {epoch + 1}/50 - loss: {history["loss"][-1]:.4f} - mae: {history["mae"][-1]:.4f} - val_loss: {val_loss:.4f} - val_mae: {val_mae:.4f}')

print('\nSTEP 6: Testing on validation set...\n')
with torch.no_grad():
    Y_pred_norm = training_model(X_val_t).cpu().numpy()
Y_pred = Y_pred_norm * Y_sigma + Y_mu
errors = np.abs(Y_pred - Y_val)
print('Mean absolute errors:')
print(f'  a (semi-axis1): {np.mean(errors[:, 0]):.3f}')
print(f'  b (semi-axis2): {np.mean(errors[:, 1]):.3f}')
print(f'  theta (angle): {np.degrees(np.mean(errors[:, 2])):.2f} degrees')


## Validation Summary

In [ ]:
error_summary = pd.DataFrame({
    "Parameter": ["a", "b", "theta"],
    "Mean Absolute Error": [
        float(np.mean(errors[:, 0])),
        float(np.mean(errors[:, 1])),
        float(np.degrees(np.mean(errors[:, 2]))),
    ],
    "Units": ["length", "length", "degrees"],
})

error_summary

## Student Exercise

1. Change the network architecture (add/remove layers)
2. Increase `N` and see how error changes
3. Add noise to inputs and observe performance drop